In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import os 

BASE_DIR = os.getcwd()

TEST1_S22U = [
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_looking_left.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_looking_right.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_swing_left.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_swing_right.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_calling_left.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S22U_calling_right.csv")
]

TEST1_S20P = [
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_looking_left.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_looking_right.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_swing_left.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_swing_right.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_calling_left.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test1_S20P_calling_right.csv")
]

TEST2_S22U = [
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_looking.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_swing.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S22U_calling.csv")
]

TEST2_S20P = [
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_looking.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_swing.csv"),
    os.path.join(BASE_DIR, "results/predicted_trajectory/test2_S20P_calling.csv")
]

CONV_PDR_PATHS = [
    os.path.join(BASE_DIR, "results/conv/test1_looking_left.csv"),
    os.path.join(BASE_DIR, "results/conv/test1_looking_right.csv"),
    os.path.join(BASE_DIR, "results/conv/test1_swing_left.csv"),
    os.path.join(BASE_DIR, "results/conv/test1_swing_right.csv"),
    os.path.join(BASE_DIR, "results/conv/test1_calling_left.csv"),
    os.path.join(BASE_DIR, "results/conv/test1_calling_right.csv"),
    os.path.join(BASE_DIR, "results/conv/test2_looking.csv"),
    os.path.join(BASE_DIR, "results/conv/test2_swing.csv"),
    os.path.join(BASE_DIR, "results/conv/test2_calling.csv")
]

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 1. 기준값 설정
# ============================================================

# 종료점 기준 좌표
# 현재는 모든 테스트가 시작점으로 복귀한다고 가정
TRUE_END_POINTS = {
    "test1": (0.0, 0.0),
    "test2": (0.0, 0.0),
}

# 누적 Heading 기준값
TARGET_HEADINGS = {
    "test1_left": -1080.0,
    "test1_right": 1080.0,
    "test2": 0.0,
}


# ============================================================
# 2. 경로 CSV 로드 함수
# ============================================================

def load_trajectory(csv_path):
    df = pd.read_csv(csv_path)

    required_cols = ["x", "y", "heading"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"{csv_path} 파일에 '{col}' 컬럼이 없습니다.")

    return df


# ============================================================
# 3. 테스트 정보 파싱 함수
# ============================================================

def parse_test_info(file_path):
    """
    파일명 예시:
    - test1_S22U_looking_left.csv
    - test1_S20P_swing_right.csv
    - test2_S22U_looking.csv
    - test1_looking_left.csv       # Conv PDR
    - test2_looking.csv            # Conv PDR
    """

    name = Path(file_path).stem

    if name.startswith("test1"):
        testbed = "test1"
    elif name.startswith("test2"):
        testbed = "test2"
    else:
        testbed = "unknown"

    if "S22U" in name:
        device = "S22U"
    elif "S20P" in name:
        device = "S20+"
    else:
        device = "-"

    if "looking" in name:
        motion = "Looking"
    elif "swing" in name:
        motion = "Swing"
    elif "calling" in name:
        motion = "Calling"
    else:
        motion = "Unknown"

    if "left" in name:
        turn = "Left"
    elif "right" in name:
        turn = "Right"
    else:
        turn = "-"

    return testbed, device, motion, turn


# ============================================================
# 4. 기준 Heading 반환 함수
# ============================================================

def get_target_heading(testbed, turn):
    if testbed == "test1":
        if turn == "Left":
            return TARGET_HEADINGS["test1_left"]
        elif turn == "Right":
            return TARGET_HEADINGS["test1_right"]
        else:
            raise ValueError("test1 데이터인데 left/right 정보가 없습니다.")

    elif testbed == "test2":
        return TARGET_HEADINGS["test2"]

    else:
        raise ValueError("알 수 없는 testbed입니다.")


# ============================================================
# 5. 단일 경로 오차 계산 함수
# ============================================================

def compute_trajectory_error(csv_path, method):
    df = load_trajectory(csv_path)

    testbed, device, motion, turn = parse_test_info(csv_path)

    true_end_x, true_end_y = TRUE_END_POINTS[testbed]

    pred_end_x = df["x"].iloc[-1]
    pred_end_y = df["y"].iloc[-1]

    final_position_error = np.sqrt(
        (pred_end_x - true_end_x) ** 2 +
        (pred_end_y - true_end_y) ** 2
    )

    pred_final_heading = np.degrees(df["heading"].iloc[-1])
    target_heading = get_target_heading(testbed, turn)

    heading_error = abs(abs(pred_final_heading) - abs(target_heading))

    result = {
        "Testbed": testbed,
        "Device": device,
        "Motion": motion,
        "Turn": turn,
        "Method": method,
        "Pred End X (m)": pred_end_x,
        "Pred End Y (m)": pred_end_y,
        "True End X (m)": true_end_x,
        "True End Y (m)": true_end_y,
        "Final Position Error (m)": final_position_error,
        "Pred Heading (deg)": pred_final_heading,
        "Target Heading (deg)": target_heading,
        "Heading Error (deg)": heading_error,
        "File": Path(csv_path).name,
    }

    return result


# ============================================================
# 6. 모든 파일 분석
# ============================================================

results = []

# Proposed AI-PDR
for path in TEST1_S22U:
    if os.path.exists(path):
        results.append(compute_trajectory_error(path, method="Proposed"))
    else:
        print(f"파일 없음: {path}")

for path in TEST1_S20P:
    if os.path.exists(path):
        results.append(compute_trajectory_error(path, method="Proposed"))
    else:
        print(f"파일 없음: {path}")

for path in TEST2_S22U:
    if os.path.exists(path):
        results.append(compute_trajectory_error(path, method="Proposed"))
    else:
        print(f"파일 없음: {path}")

for path in TEST2_S20P:
    if os.path.exists(path):
        results.append(compute_trajectory_error(path, method="Proposed"))
    else:
        print(f"파일 없음: {path}")


# Conventional PDR
for path in CONV_PDR_PATHS:
    if os.path.exists(path):
        results.append(compute_trajectory_error(path, method="Conventional PDR"))
    else:
        print(f"파일 없음: {path}")


# ============================================================
# 7. 결과 DataFrame 생성 및 저장
# ============================================================

result_df = pd.DataFrame(results)

SAVE_DIR = os.path.join(BASE_DIR, "result", "analysis")
os.makedirs(SAVE_DIR, exist_ok=True)

save_path = os.path.join(SAVE_DIR, "trajectory_error_summary.csv")
result_df.to_csv(save_path, index=False, encoding="utf-8-sig")

print(f"분석 결과 저장 완료: {save_path}")

result_df

분석 결과 저장 완료: d:\NNL\AI-PDR\AI-PDR_GPS-GT\result\analysis\trajectory_error_summary.csv


,Testbed,Device,Motion,Turn,Method,Pred End X (m),Pred End Y (m),True End X (m),True End Y (m),Final Position Error (m),Pred Heading (deg),Target Heading (deg),Heading Error (deg),File
0,test1,S22U,Looking,Left,Proposed,0.535298,-1.221452,0.0,0.0,1.333600,1090.155630,-1080.0,10.155630,test1_S22U_looking_left.csv
1,test1,S22U,Looking,Right,Proposed,1.224690,0.316306,0.0,0.0,1.264878,-1064.882823,1080.0,15.117177,test1_S22U_looking_right.csv
2,test1,S22U,Swing,Left,Proposed,-3.863214,2.613953,0.0,0.0,4.664459,1050.346368,-1080.0,29.653632,test1_S22U_swing_left.csv
3,test1,S22U,Swing,Right,Proposed,5.072003,7.975323,0.0,0.0,9.451508,-1036.299830,1080.0,43.700170,test1_S22U_swing_right.csv
4,test1,S22U,Calling,Left,Proposed,-2.759671,0.554297,0.0,0.0,2.814787,1069.864559,-1080.0,10.135441,test1_S22U_calling_left.csv
5,test1,S22U,Calling,Right,Proposed,2.844915,-0.225829,0.0,0.0,2.853864,-1070.332607,1080.0,9.667393,test1_S22U_calling_right.csv
6,test1,S20+,Looking,Left,Proposed,0.972750,-0.771534,0.0,0.0,1.241574,1099.881150,-1080.0,19.881150,test1_S20P_looking_left.csv
7,test1,S20+,Looking,Right,Proposed,-0.492248,-0.391065,0.0,0.0,0.628682,-1067.660496,1080.0,12.339504,test1_S20P_looking_right.csv
8,test1,S20+,Swing,Left,Proposed,-7.580349,2.343795,0.0,0.0,7.934423,1025.603312,-1080.0,54.396688,test1_S20P_swing_left.csv
9,test1,S20+,Swing,Right,Proposed,4.578570,9.817116,0.0,0.0,10.832316,-1023.272188,1080.0,56.727812,test1_S20P_swing_right.csv
